# ML similitud del coseno

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import LabelEncoder

## EDA preliminar

En primer lugar se realizar? un an?lisis exploratorio para poder reducir los datos debido a los recursos del entorno donde se hace el despliegue de la aplicaci?n 

In [ ]:
#Se cargan los datos
df_games=pd.read_parquet('Datasets/data_ML.parquet')

In [ ]:
df_users=pd.read_parquet('Datasets/users_items_clean.parquet')

In [ ]:
df_games['release_year']=df_games['release_year'].astype('int16')

In [ ]:
df_games.info()

In [ ]:
df_users.info()

In [ ]:
df_most_popular=pd.merge(df_games,df_users[['item_id','playtime_forever']],left_on='id',right_on='item_id',how='inner')

In [ ]:
df_most_popular.drop(columns=['item_id','id'],inplace=True)

Filtro los juegos por los que se lanzaron a partir del 2014.

In [ ]:
df_most_popular=df_most_popular[df_most_popular['release_year']>=2014]

In [ ]:
df_playtime=df_most_popular.groupby('app_name')['playtime_forever'].sum().reset_index()

In [ ]:
df_playtime.sort_values(by='playtime_forever',ascending=False)

In [ ]:
df_playtime=df_playtime[df_playtime['playtime_forever']>0]

In [ ]:
df_playtime

In [ ]:
data=pd.merge(df_playtime['app_name'],df_games,left_on='app_name',right_on='app_name',how='inner')

In [ ]:
data.info()

In [ ]:
data.drop('release_year',axis=1,inplace=True)

In [ ]:
data.to_parquet('Datasets/data.parquet')

## Modelo de la similitud del coseno

In [ ]:
df=pd.read_parquet('Datasets/data.parquet')

In [ ]:
df.info()

In [ ]:
df.reset_index(drop=True, inplace=True)

In [ ]:
#Vectorizar el nombre de los productos
vectorizer=TfidfVectorizer()
tfidf_matrix=vectorizer.fit_transform(df[['app_name','genres','tags']].apply(lambda x: ' '.join(x),axis=1))

In [ ]:
# Calcular la cantidad de memoria en bytes
memory_in_bytes = (tfidf_matrix.data.nbytes + tfidf_matrix.indices.nbytes + tfidf_matrix.indptr.nbytes)

# Convertir a megabytes
memory_in_megabytes = memory_in_bytes / (1024 * 1024)

print(f"Memory: {memory_in_megabytes} MB")

In [ ]:
#Combinar caracter?sticas
#features=np.column_stack([tfidf_matrix.toarray(),df['genres'],df['tags']])

In [ ]:
#Calcular la matriz de similitud del coseno
similarity_matrix=cosine_similarity(tfidf_matrix)

In [ ]:

# Calcular la cantidad de memoria en bytes
memory_in_bytes = similarity_matrix.nbytes

# Convertir a megabytes
memory_in_megabytes = memory_in_bytes / (1024 * 1024)

print(f"Memory: {memory_in_megabytes} MB")

In [ ]:
df

In [ ]:

item_id=508290
if df[df['id']==item_id].empty:
    print('No existe el item')
else:
    producto_index=df[df['id']==item_id].index[0]
    product_similarities=similarity_matrix[producto_index]
    most_similar_products_index=np.argsort(-product_similarities)[1:6]
    most_similar_products=df.loc[most_similar_products_index,'app_name']
    print(most_similar_products)

In [ ]:
most_similar_products.values.tolist()

In [ ]:
most_similar_products.to_json(orient='records')

In [ ]:
{"Recomendaci?n": most_similar_products.to_json(orient='records')}